### RAG Pipeline - Data Ingestion to Vector DB Pipeline


In [14]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from pathlib import Path 



/var/folders/q7/7y8vj3_93qz3wsbyfjkj51d80000gn/T/ipykernel_42989/283692175.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [15]:
### Read all PDFs inside the directory. 

def process_all_pdfs(pdf_directory): 
    """Process all PDFs` in a directory."""
    all_documents = []
    pdf_dir = Path(pdf_directory) 

    # Find all PDF files recursively. 
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nprocessing:{pdf_file.name}")
        try:
            loader= PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata  
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type]'] = 'pdf'

                all_documents.extend(documents)
                print(f"✅ loaded {len(documents)} pages") 

        except Exception as e:
            print(f" Error: {e}")

    print(f"\n total documents loaded: {len(all_documents)}")  
    return all_documents 

# All PDFs in the data directory 
all_pdfs_documents = process_all_pdfs("../data/PDFLearn")


Found 8 PDF files to process

processing:JobRadar_Build_Guide.pdf
✅ loaded 6 pages
✅ loaded 6 pages
✅ loaded 6 pages
✅ loaded 6 pages
✅ loaded 6 pages
✅ loaded 6 pages

processing:10_10 Engineer’s Resource Guide _ Vishakha Sadhwani.pdf
✅ loaded 2 pages
✅ loaded 2 pages

processing:ML-system-design-interviews.pdf
✅ loaded 1 pages

processing:100 LLM Interview Questions .pdf
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pages
✅ loaded 107 pag

In [16]:
all_pdfs_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-11T13:07:42+00:00', 'author': '', 'keywords': '', 'moddate': '2026-04-11T13:07:42+00:00', 'subject': '(unspecified)', 'title': 'JobRadar Build Guide', 'trapped': '/False', 'source': '../data/PDFLearn/JobRadar_Build_Guide.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'JobRadar_Build_Guide.pdf', 'file_type]': 'pdf'}, page_content='JobRadar\nHow to build your own AI-powered job hunting app with no code\nThis is the exact process I used to build JobRadar, a personalised job hunting app that parses your CV, pulls\nlive jobs from LinkedIn, scores each one against your profile, and tracks your applications in a Kanban board.\nI built it using Emergent as the app builder, Groq for the LLM calls, and Apify to scrape LinkedIn jobs.\nYou do not need to write any backend or frontend code. Just paste the prompts below into Emergent in order.\nYou can al

In [17]:
from pathlib import Path

from langchain_community.document_loaders import (
    DirectoryLoader,
    PyMuPDFLoader,
    UnstructuredImageLoader,
)


def load_documents(data_dir: str):

    data_path = Path(data_dir)

    all_documents = []

    # ========================================================
    # Load PDFs
    # ========================================================

    print("\n" + "=" * 80)
    print("LOADING PDF DOCUMENTS")
    print("=" * 80)

    pdf_loader = DirectoryLoader(
        str(data_path),
        glob="**/*.pdf",
        loader_cls=PyMuPDFLoader,
        show_progress=True,
        silent_errors=True,
    )

    try:
        pdf_documents = pdf_loader.load()

        # Add standardized metadata
        for doc in pdf_documents:

            source = doc.metadata.get("source")

            if source:
                doc.metadata["source_file"] = Path(source).name

            doc.metadata["file_type"] = "pdf"

        all_documents.extend(pdf_documents)

        print(f"\nPDF documents loaded: {len(pdf_documents)}")

    except Exception as e:

        print(f"\n❌ Error loading PDFs: {e}")


    # ========================================================
    # Load Images
    # ========================================================

    print("\n" + "=" * 80)
    print("LOADING IMAGE DOCUMENTS")
    print("=" * 80)

    image_extensions = [
        "*.png",
        "*.jpg",
        "*.jpeg",
        "*.tiff",
        "*.tif",
        "*.bmp",
        "*.webp",
    ]

    for extension in image_extensions:

        print(f"\nSearching for: {extension}")

        image_loader = DirectoryLoader(
            str(data_path),
            glob=f"**/{extension}",
            loader_cls=UnstructuredImageLoader,
            loader_kwargs={
                "mode": "elements",
            },
            show_progress=True,
            silent_errors=True,
        )

        try:

            image_documents = image_loader.load()

            # Add standardized metadata
            for doc in image_documents:

                source = doc.metadata.get("source")

                if source:
                    doc.metadata["source_file"] = Path(source).name

                doc.metadata["file_type"] = "image"

            all_documents.extend(image_documents)

            print(
                f"Loaded {len(image_documents)} "
                f"documents from {extension}"
            )

        except Exception as e:

            print(
                f"❌ Error loading {extension}: {e}"
            )


    # ========================================================
    # Final Summary
    # ========================================================

    print("\n" + "=" * 80)
    print("DOCUMENT INGESTION COMPLETE")
    print("=" * 80)

    print(
        f"Total documents loaded: "
        f"{len(all_documents)}"
    )

    return all_documents


# ============================================================
# Load everything
# ============================================================

documents = load_documents(
    "../data/PDFLearn"
)


# ============================================================
# Inspect documents
# ============================================================

print("\n" + "=" * 80)
print("DOCUMENT INSPECTION")
print("=" * 80)


for i, doc in enumerate(documents[:10]):

    print("\n" + "-" * 80)

    print(f"Document {i + 1}")

    print("-" * 80)

    print(
        "Source:",
        doc.metadata.get("source_file")
    )

    print(
        "File type:",
        doc.metadata.get("file_type")
    )

    print(
        "Page:",
        doc.metadata.get("page")
    )

    print(
        "Content:"
    )

    print(
        doc.page_content[:500]
    )


LOADING PDF DOCUMENTS


100%|██████████| 8/8 [00:01<00:00,  7.95it/s]



PDF documents loaded: 341

LOADING IMAGE DOCUMENTS

Searching for: *.png


0it [00:00, ?it/s]


Loaded 0 documents from *.png

Searching for: *.jpg


...

Loaded 0 documents from *.tif

Searching for: *.bmp


0it [00:00, ?it/s]


Loaded 0 documents from *.bmp

Searching for: *.webp


0it [00:00, ?it/s]

Loaded 0 documents from *.webp

DOCUMENT INGESTION COMPLETE
Total documents loaded: 385

DOCUMENT INSPECTION

--------------------------------------------------------------------------------
Document 1
--------------------------------------------------------------------------------
Source: JobRadar_Build_Guide.pdf
File type: pdf
Page: 0
Content:
JobRadar
How to build your own AI-powered job hunting app with no code
This is the exact process I used to build JobRadar, a personalised job hunting app that parses your CV, pulls
live jobs from LinkedIn, scores each one against your profile, and tracks your applications in a Kanban board.
I built it using Emergent as the app builder, Groq for the LLM calls, and Apify to scrape LinkedIn jobs.
You do not need to write any backend or frontend code. Just paste the prompts below into Emergent in or

--------------------------------------------------------------------------------
Document 2
----------------------------------------------------------

In [18]:
all_documents

NameError: name 'all_documents' is not defined

In [19]:
### Text spllitting get into chunks

def split_documents(documents,chunk_size = 1000,chunk_overlap = 200):
    """Split documents into smaller chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter( 
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap, 
        length_function=len, 
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)  
    print(f"split {len(documents)} documents into {len(split_docs)} chunks") 

    # Show an example of a chunk. 
    if split_docs: 
        print(f"\nExample chunk:") 
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadat: {split_docs[0].metadata}")

    return split_docs

In [20]:
chunks = split_documents(documents)
chunks

split 385 documents into 377 chunks

Example chunk:
Content: JobRadar
How to build your own AI-powered job hunting app with no code
This is the exact process I used to build JobRadar, a personalised job hunting app that parses your CV, pulls
live jobs from Link...
Metadat: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-11T13:07:42+00:00', 'source': '../data/PDFLearn/JobRadar_Build_Guide.pdf', 'file_path': '../data/PDFLearn/JobRadar_Build_Guide.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'JobRadar Build Guide', 'author': '', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-04-11T13:07:42+00:00', 'trapped': '', 'modDate': "D:20260411130742+00'00'", 'creationDate': "D:20260411130742+00'00'", 'page': 0, 'source_file': 'JobRadar_Build_Guide.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-11T13:07:42+00:00', 'source': '../data/PDFLearn/JobRadar_Build_Guide.pdf', 'file_path': '../data/PDFLearn/JobRadar_Build_Guide.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'JobRadar Build Guide', 'author': '', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-04-11T13:07:42+00:00', 'trapped': '', 'modDate': "D:20260411130742+00'00'", 'creationDate': "D:20260411130742+00'00'", 'page': 0, 'source_file': 'JobRadar_Build_Guide.pdf', 'file_type': 'pdf'}, page_content='JobRadar\nHow to build your own AI-powered job hunting app with no code\nThis is the exact process I used to build JobRadar, a personalised job hunting app that parses your CV, pulls\nlive jobs from LinkedIn, scores each one against your profile, and tracks your applications in a Kanban board.\nI built it using Emergent as the app builder, Groq for the LLM calls, and Apify to scrape L

# Embedding and Vector Store DB

In [21]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [102]:
class EmbeddingManager:
    """ Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLm-L6-v2"):
        """
        Initialize the embedding manager.
        
        Args:
        Model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name 
        self.model = None 
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e: 
            print(f"Error loading model {self.model_name}: {e}")
            raise 

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Get embeddings for a list of text.
        
        Args:
            Text: list of text strings to embed 

        Return: 
            numpy array of embeddings with shape (len(texts), embedding_dim) 
        """  
        if not self.model:
            raise ValueError("Model not loaded")       
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings     


### Initialize the embedding manager

embedding_manager=EmbeddingManager() 
embedding_manager

Loading embedding model: all-MiniLm-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimensions: 384


### Vector Store

In [103]:
import os
import uuid
import numpy as np
import chromadb

from typing import Any


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "learn_documents",
        persistent_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection
            persistent_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:

            # Create a persistent ChromaDB client
            os.makedirs(
                self.persistent_directory,
                exist_ok=True
            )

            self.client = chromadb.PersistentClient(
                path=self.persistent_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF and image document embeddings for RAG"
                }
            )

            print(
                f"`VectorStore` initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:

            print(
                f"Error initiating vector store: {e}"
            )

            raise

    def add_documents(
        self,
        documents: list[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the VectorStore.

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):

            raise ValueError(
                "Number of documents must match "
                "number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents "
            f"to vector store..."
        )

        # --------------------------------------------------------
        # Prepare data for ChromaDB
        # --------------------------------------------------------

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        # --------------------------------------------------------
        # Process each document
        # --------------------------------------------------------

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique ID
            doc_id = (
                f"doc_{uuid.uuid4().hex[:8]}_{i}"
            )

            ids.append(doc_id)

            # ----------------------------------------------------
            # Prepare metadata
            #
            # IMPORTANT:
            # ChromaDB metadata values must be simple types:
            # str, int, float, bool, list, or None.
            #
            # We therefore explicitly select the metadata fields
            # we want to store instead of copying the entire
            # doc.metadata dictionary.
            # ----------------------------------------------------

            metadata = {
                "source_file": str(
                    doc.metadata.get(
                        "source_file",
                        ""
                    )
                ),

                "file_type": str(
                    doc.metadata.get(
                        "file_type",
                        ""
                    )
                ),

                "page": int(
                    doc.metadata.get(
                        "page",
                        -1
                    )
                ),

                "doc_index": i,

                "content_length": len(
                    doc.page_content
                )
            }

            metadatas.append(metadata)

            # ----------------------------------------------------
            # Document content
            # ----------------------------------------------------

            documents_text.append(
                doc.page_content
            )

            # ----------------------------------------------------
            # Embedding
            # ----------------------------------------------------

            embeddings_list.append(
                embedding.tolist()
            )

        # --------------------------------------------------------
        # Add everything to ChromaDB
        # --------------------------------------------------------

        try:

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added "
                f"{len(documents)} documents "
                f"to vector store"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:

            print(
                f"Error adding documents "
                f"to vector store: {e}"
            )

            raise


    

In [104]:
vectorstore=VectorStore()
vectorstore

`VectorStore` initialized. Collection: learn_documents
Existing documents in collection: 754


In [105]:
import os

print(os)
print(os.getcwd())

<module 'os' (frozen)>
/Users/dludhani/Projects/RAG_Learn/notebook


In [106]:
import os
import uuid
import numpy as np
import chromadb

from typing import Any


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "learn_documents",
        persistent_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection
            persistent_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:

            # Create directory if it does not exist
            os.makedirs(
                self.persistent_directory,
                exist_ok=True
            )

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persistent_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": (
                        "PDF and image document embeddings for RAG"
                    )
                }
            )

            print(
                f"`VectorStore` initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:

            print(
                f"Error initiating vector store: {e}"
            )

            raise

    def add_documents(
        self,
        documents: list[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the VectorStore.

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings
            for the documents
        """

        # --------------------------------------------------------
        # Validate number of documents and embeddings
        # --------------------------------------------------------

        if len(documents) != len(embeddings):

            raise ValueError(
                "Number of documents must match "
                "number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents "
            f"to vector store..."
        )

        # --------------------------------------------------------
        # Prepare data for ChromaDB
        # --------------------------------------------------------

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        # --------------------------------------------------------
        # Process each document
        # --------------------------------------------------------

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # ----------------------------------------------------
            # Generate unique ID
            # ----------------------------------------------------

            doc_id = (
                f"doc_{uuid.uuid4().hex[:8]}_{i}"
            )

            ids.append(doc_id)

            # ----------------------------------------------------
            # Prepare ONLY ChromaDB-compatible metadata
            # ----------------------------------------------------

            source_file = doc.metadata.get(
                "source_file",
                ""
            )

            file_type = doc.metadata.get(
                "file_type",
                ""
            )

            page = doc.metadata.get(
                "page",
                -1
            )

            # ----------------------------------------------------
            # Safely convert metadata values
            # ----------------------------------------------------

            if source_file is None:
                source_file = ""

            else:
                source_file = str(
                    source_file
                )

            if file_type is None:
                file_type = ""

            else:
                file_type = str(
                    file_type
                )

            # Page may sometimes be None or a non-integer
            try:

                if page is None:
                    page = -1

                else:
                    page = int(page)

            except (TypeError, ValueError):

                page = -1

            # ----------------------------------------------------
            # Create clean metadata
            # ----------------------------------------------------

            metadata = {
                "source_file": source_file,
                "file_type": file_type,
                "page": page,
                "doc_index": int(i),
                "content_length": int(
                    len(doc.page_content)
                )
            }

            # ----------------------------------------------------
            # DEBUG
            # ----------------------------------------------------

            print(
                f"\nDocument {i + 1} metadata:"
            )

            print(metadata)

            # Make absolutely sure there are no dictionaries
            # or other unsupported objects in metadata

            for key, value in metadata.items():

                if isinstance(value, dict):

                    raise ValueError(
                        f"Invalid dictionary metadata "
                        f"found for key '{key}': {value}"
                    )

            metadatas.append(metadata)

            # ----------------------------------------------------
            # Document content
            # ----------------------------------------------------

            documents_text.append(
                str(doc.page_content)
            )

            # ----------------------------------------------------
            # Embedding
            # ----------------------------------------------------

            embeddings_list.append(
                embedding.tolist()
            )

        # --------------------------------------------------------
        # Add everything to ChromaDB
        # --------------------------------------------------------

        try:

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"\nSuccessfully added "
                f"{len(documents)} documents "
                f"to vector store"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:

            print(
                "\n❌ Error adding documents "
                f"to vector store: {e}"
            )

            # Print the metadata that was actually
            # sent to ChromaDB

            print(
                "\nMetadata sent to ChromaDB:"
            )

            for i, metadata in enumerate(
                metadatas
            ):

                print(
                    f"Document {i}: {metadata}"
                )

            raise

In [107]:
vectorstore = VectorStore()

`VectorStore` initialized. Collection: learn_documents
Existing documents in collection: 754


In [108]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-11T13:07:42+00:00', 'source': '../data/PDFLearn/JobRadar_Build_Guide.pdf', 'file_path': '../data/PDFLearn/JobRadar_Build_Guide.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': 'JobRadar Build Guide', 'author': '', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-04-11T13:07:42+00:00', 'trapped': '', 'modDate': "D:20260411130742+00'00'", 'creationDate': "D:20260411130742+00'00'", 'page': 0, 'source_file': 'JobRadar_Build_Guide.pdf', 'file_type': 'pdf'}, page_content='JobRadar\nHow to build your own AI-powered job hunting app with no code\nThis is the exact process I used to build JobRadar, a personalised job hunting app that parses your CV, pulls\nlive jobs from LinkedIn, scores each one against your profile, and tracks your applications in a Kanban board.\nI built it using Emergent as the app builder, Groq for the LLM calls, and Apify to scrape L

In [109]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts

['JobRadar\nHow to build your own AI-powered job hunting app with no code\nThis is the exact process I used to build JobRadar, a personalised job hunting app that parses your CV, pulls\nlive jobs from LinkedIn, scores each one against your profile, and tracks your applications in a Kanban board.\nI built it using Emergent as the app builder, Groq for the LLM calls, and Apify to scrape LinkedIn jobs.\nYou do not need to write any backend or frontend code. Just paste the prompts below into Emergent in order.\nYou can also modify them for your own role, tools, or design preferences. I have noted where changes make\nsense.\nWhat you need before you start\nEmergent\nThe AI app builder used to generate the entire app from prompts. Sign up at emergent.sh\nand create a new project.\nGroq API key\nUsed for CV parsing and job match scoring. Get yours at console.groq.com under API\nKeys.\nApify API key\nUsed to scrape live LinkedIn job listings. Sign up at apify.com, go to Settings and then',
 'U

In [110]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

### Generate the Embedding

embeddings=embedding_manager.generate_embeddings(texts)

### Store in the Vector DB
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 377 texts...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Generated embeddings with shape: (377, 384)
Adding 377 documents to vector store...

Document 1 metadata:
{'source_file': 'JobRadar_Build_Guide.pdf', 'file_type': 'pdf', 'page': 0, 'doc_index': 0, 'content_length': 976}

Document 2 metadata:
{'source_file': 'JobRadar_Build_Guide.pdf', 'file_type': 'pdf', 'page': 0, 'doc_index': 1, 'content_length': 477}

Document 3 metadata:
{'source_file': 'JobRadar_Build_Guide.pdf', 'file_type': 'pdf', 'page': 1, 'doc_index': 2, 'content_length': 966}

Document 4 metadata:
{'source_file': 'JobRadar_Build_Guide.pdf', 'file_type': 'pdf', 'page': 1, 'doc_index': 3, 'content_length': 998}

Document 5 metadata:
{'source_file': 'JobRadar_Build_Guide.pdf', 'file_type': 'pdf', 'page': 1, 'doc_index': 4, 'content_length': 437}

Document 6 metadata:
{'source_file': 'JobRadar_Build_Guide.pdf', 'file_type': 'pdf', 'page': 2, 'doc_index': 5, 'content_length': 948}

Document 7 metadata:
{'source_file': 'JobRadar_Build_Guide.pdf', 'file_type': 'pdf', 'page': 3, 'do

# Retriever Pipeline From VectorStore

In [111]:
class RAGRetriever:
    """ Handles query was retrieval from the vector store. """

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever 

        Args:
            vector_store: Vector store containing document embeddings 
            embedding_manager: Manager for generating query embeddings 
        """
        self._vector_store = vector_store 
        self.embedding_manager = embedding_manager 

    def retrieve(self, query: str, top_k: int=5, score_threshold: float= 0.0) -> list[Dict[str, Any]]:
        """ 
        Retrieve relevant documents for a query. 
        Args: 
            query: The search query 
            top_k: Number of top results to return 
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata    
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")  

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self._vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB use cosine distance)  
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })  

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found") 

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [112]:
rag_retriever

In [113]:
rag_retriever.retrieve("Why is chunking necessary?")

Retrieving documents for query: 'Why is chunking necessary?'
Top k: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_1422216e_127',
  'content': 'Q67. What is Chunking?\nIntermediate\nInterview Answer\nChunking is the process of dividing large documents into smaller pieces before embedding and indexing them\nfor retrieval.\nWhy do we chunk?\nSuppose you have a 100-page PDF — you don\'t usually want to embed the entire document as one giant piece.\nInstead, each chunk can be embedded separately, so when a user asks a question, the system can retrieve only the\nrelevant chunks.\nExample\nA leave-policy section becomes a chunk; a query like "How many vacation days do employees get?" can retrieve that\nchunk.\nKey idea\nChunking converts large documents into retrievable units.\nQ68. Why is Chunking important in RAG?\nIntermediate\nInterview Answer\nChunking is important because retrieval quality depends heavily on the size and content of the units being\nsearched. Good chunks contain enough context to answer a question while remaining focused enough to\nretrieve accurately.\nToo large vs too

# Integration Vector DB Context pipeline with LLM output 

In [ ]:
import os
from dotenv import load_dotenv

# Load .env
load_dotenv()

# Check whether Groq API key exists
groq_api_key = os.getenv("GROQ_API_KEY")

if groq_api_key:
    print("GROQ_API_KEY found")
    print("Key starts with:", groq_api_key[:8])
    print("Key length:", len(groq_api_key))
else:
    print("❌ GROQ_API_KEY not found")

In [ ]:
### Simple RAG pipeline with Grok LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initializa the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name='qwen/qwen3.6-27b',temperature=0.1,max_tokens=1024)

# Simple RAG function: retrieve content + generate response
def rag_simple (query, retriever,llm,top_k=10):
    # retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content']for doc in results]) if results else ""
    if not context:
        return "No relevantcontext found to answer the question."

    ## Generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer: """
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content




In [130]:
answer=rag_simple("What are the popular Agentic AI tools?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What are the popular Agentic AI tools?'
Top k: 10, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 10 documents (after filtering)
**Popular Agentic AI tools**

- **LangChain**  
- **LlamaIndex**  
- **AutoGen** (Microsoft)  
- **OpenAI’s function‑calling** capability  
- **CrewAI**  
- **AutoGPT**  
- **LangSmith** (for debugging and tracing)


In [ ]:
### Simple RAG pipeline with Grok LLM
""" if groq_llm:
    query = "Why is chunking necessary?"
    retrieved_docs = rag_retriever.retrieve(query, top_k=3, score_threshold=0.1)

    if retrieved_docs: 
        # Combine top K retrieved documents as context. 
        combined_context = "\n\n".join([doc['content'] for doc in retrieved_docs])

        # Generate responses using Groq LLM
        response= groq_llm.generate_response(query, combined_context)
        print(f"\nResponse: \n{Response}")
    else:
        print("No relevant documents found for the query.") 
"""

In [139]:
# --- Enhanced RAG pipeline features ---

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """ 
    RAG pipeline with extra features: 
    - Returns answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page','unknown'),
        'source': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    }for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer."""
    response = llm.invoke([prompt.format(context=context, query=query)])


    output = {
        'answer': response.content, 
        'sources': sources, 
        'confidence': confidence
    }
    if return_context: 
        output['context'] = context 
    return output

# Example usage
result = rag_advanced("Explain the end-to-end process of building an AI Agent project?", rag_retriever, llm, top_k=10, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'Explain the end-to-end process of building an AI Agent project?'
Top k: 10, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 10 documents (after filtering)
Answer: **End‑to‑end process for building an AI‑Agent project**

1. **Define the task & goals** – Clearly state what problem the agent must solve and the success criteria (e.g., “answer customer support queries within 2 seconds with ≥ 90 % accuracy”).

2. **Choose the LLM** – Select a large‑language model that has the required capabilities (size, knowledge domain, tool‑calling support, latency, cost).

3. **Set up memory & context** – Design how the agent will retain short‑term (conversation) and long‑term (user profile, past interactions) memory, and configure access to external resources such as APIs, databases, or knowledge bases.

4. **Prompt engineering & reasoning** – Craft system and user prompts, embed chain‑of‑thought (CoT) or other reasoning patterns, and define any role‑playing or instruction templates that guide the model’s behavior.

5. **Integrate tool use (function calling)** – Implement t